# YOLO Training

In [1]:
%pip install pytz pandas 

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
from logger import get_logger
logger = get_logger("YOLO-TRAIN")

2025-09-10 13:51 - INFO - Logger YOLO-TRAIN iniciado


## Environment

### Verificar CUDA en env

In [ ]:
import torch

torch.cuda.is_available()

True

### Verificar librería ultralytics

In [4]:
# Issue por compatibilidad !!! 8.3.80
%pip install -qU ultralytics==8.3.80

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from ultralytics import __version__ as ultralytics_version

ultralytics_version

'8.3.80'

### Importar dependencias

Ingresamos la ruta del dataset

In [ ]:
import os
import pandas as pd
from datetime import datetime
from ultralytics import YOLO

DATASET_PATH = os.path.join(os.getcwd(), "data/desmodus-rotundus-1.v8i.yolov11")

## YOLO series

Definimos los métodos de entrenamiento y exportación a formatos PT (PYTORCH) y TFLITE

Los hiperparámetros definidos son: 
device="cuda",
imgsz=640,
batch=16,
workers=64,
epochs=50,
pretrained=False,

In [ ]:
def train_yolo_model(model: YOLO, seed: int = 0):
    """Entrena el modelo yolo con el dataset de lissachatina"""
    res = model.train(
        data=os.path.join(DATASET_PATH, "data.yaml"),
        # optimizer="auto", # SGD, Adam, AdamW, NAdam, RAdam, RMSProp etc., or auto
        # lr0=0.01, #  (i.e. SGD=1E-2, Adam=1E-3)
        seed=seed,
        close_mosaic=True,
        device="cuda",
        imgsz=640,
        batch=16,
        # workers=64,
        workers=0,
        epochs=100,
        pretrained=False,
        patience=15,
    )

    return res


def export_yolo_model(model: YOLO) -> str:
    """Exporta el modelo yolo a tflite con Float16"""

    res_dir = model.export(
        format="tflite",
        half=True,
        # int8=True,
        imgsz=320,
        workers=0,
        # workers=64,
        device="cuda",
        data=os.path.join(DATASET_PATH, "data.yaml"),
    )

    return res_dir


def save_results_to_csv(trained_yolo_path: dict[tuple, str], name: str):
    """Guarda resultados de modelo, semilla y path en un csv"""
    df = pd.DataFrame(
        [
            {"yolo": yolo, "seed": seed, "path": path}
            for (yolo, seed), path in trained_yolo_path.items()
        ]
    )

    # Save to CSV
    name = name if name.endswith(".csv") else f"{name}.csv"
    df.to_csv(name, index=False)

### Train YOLO's (.pt)

In [8]:
trained_yolo_paths: dict[tuple, str] = {}

# Train for 2 different seeds
for seed in [3000]:
    for yolo in ["yolov8n", "yolov9t", "yolov10n", "yolo11n", "yolo12n"]:
        logger.info(f"Entrenando YOLO {yolo} con seed {seed}")

        yolo_model = YOLO(yolo)
        results = train_yolo_model(model=yolo_model, seed=seed)
        best_model_path = f"{str(results.save_dir)}/weights/best.pt"

        trained_yolo_paths[(yolo, seed)] = best_model_path
        logger.info("Guardado en %s", best_model_path)

TIMESTAMP = datetime.now().isoformat().replace(":", "-").replace(".", "-")
filename = f"{TIMESTAMP}_entrenamiento_yolo"
save_results_to_csv(trained_yolo_paths, filename)

logger.info("CSV file '%s' saved successfully.", filename)
trained_yolo_paths

2025-09-10 13:51 - INFO - Entrenando YOLO yolov8n con seed 3000


100%|██████████| 6.25M/6.25M [00:00<00:00, 22.5MB/s]

New https://pypi.org/project/ultralytics/8.3.198 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.80  Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24564MiB)


engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=c:\Users\ialab\Documents\dev\desmodus-app\packages\notebooks\data/desmodus-rotundus-1.v8i.yolov11\data.yaml, epochs=100, time=None, patience=15, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=0, project=None, name=train, exist_ok=False, pretrained=False, optimizer=auto, verbose=True, seed=3000, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=True, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_

100%|██████████| 5.35M/5.35M [00:00<00:00, 11.0MB/s]


...


100 epochs completed in 0.988 hours.
Optimizer stripped from runs\detect\train5\weights\last.pt, 5.5MB
Optimizer stripped from runs\detect\train5\weights\best.pt, 5.5MB

Validating runs\detect\train5\weights\best.pt...
Ultralytics 8.3.80  Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24564MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:05<00:00,  2.60it/s]


                   all        470        646      0.863      0.855      0.916      0.645
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs\detect\train5


2025-09-10 18:10 - INFO - Guardado en runs\detect\train5/weights/best.pt
2025-09-10 18:10 - INFO - CSV file '2025-09-10T18-10-20-685194_entrenamiento_yolo' saved successfully.


{('yolov8n', 3000): 'runs\\detect\\train/weights/best.pt',
 ('yolov9t', 3000): 'runs\\detect\\train2/weights/best.pt',
 ('yolov10n', 3000): 'runs\\detect\\train3/weights/best.pt',
 ('yolo11n', 3000): 'runs\\detect\\train4/weights/best.pt',
 ('yolo12n', 3000): 'runs\\detect\\train5/weights/best.pt'}

### Export YOLO's to .tflite

In [ ]:
exported_yolo_paths = {}

for (model, seed), model_path in trained_yolo_paths.items():
    logger.info(
        "### Exportando modelo %s con seed %s desde %s...", model, seed, model_path
    )

    yolo_model = YOLO(model_path)
    res_dir = export_yolo_model(model=yolo_model)

    exported_yolo_paths[(model, seed)] = res_dir
    logger.info("Exportado en %s", res_dir)

# Save to CSV
TIMESTAMP = datetime.now().isoformat()
filename = f"{TIMESTAMP}_exportado_yolo"
save_results_to_csv(exported_yolo_paths, filename)

logger.info("CSV file '%s' saved successfully.", filename)
exported_yolo_paths

2025-09-10 18:10 - INFO - ### Exportando modelo yolov8n con seed 3000 desde runs\detect\train/weights/best.pt...


Ultralytics 8.3.80  Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24564MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs\detect\train\weights\best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 5, 2100) (6.0 MB)
requirements: Ultralytics requirements ['tf_keras', 'sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'onnx>=1.12.0', 'onnx2tf>1.17.5,<=1.26.3', 'onnxslim>=0.1.31', 'tflite_support', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Retry 1/2 failed: Command 'pip install --no-cache-dir "tf_keras" "sng4onnx>=1.0.1" "onnx_graphsurgeon>=0.3.26" "onnx>=1.12.0" "onnx2tf>1.17.5,<=1.26.3" "onnxslim>=0.1.31" "tflite_support" "onnxruntime-gpu" --extra-index-url https://pypi.ngc.nvidia.com' returned non-zero exit status 1.
Retry 2/2 failed: Command 'pip install --no-cache-dir "tf_keras" "sng4onnx>=1.0.1" "onnx_graphsurgeon>=0.3.26" "onnx>=1.12.0" "onnx2tf>1.17.5,<=1.26.3" "onnxslim>=0.1.3

ModuleNotFoundError: No module named 'onnx2tf'